# 第 18 章：医学领域小模型项目

这个 notebook 对应 `lessons/18_medical_domain_project.md`，演示医学科普助手的离线闭环：可信资料 RAG、危险信号识别、用药剂量拒答、结构化输出、SFT 样本和 model card 边界。

In [ ]:
from src.medical_qa_assistant.medical_qa import (
    build_medical_sft_example,
    detect_red_flags,
    review_medical_question,
    validate_medical_citations,
    validate_medical_model_card,
)
from src.rag.baseline import (
    Document,
    HashingTextEmbedder,
    VectorStore,
    chunk_documents,
    citation_from_chunk,
)
from src.safety.governance import ModelCard

## 1. 可信医学资料

医学 RAG 资料要保留来源和适用边界；教学版用一条危险信号科普资料做最小知识库。

In [ ]:
documents = [
    Document(
        doc_id="guide",
        title="危险信号科普",
        text="胸痛、呼吸困难、意识异常属于需要及时就医或急救评估的危险信号。",
        source="trusted-reference",
    )
]
chunks = chunk_documents(documents, chunk_size=80)
store = VectorStore(chunks, HashingTextEmbedder(dim=64))
for chunk in chunks:
    print(chunk.chunk_id, chunk.text)

## 2. 危险信号识别

胸痛和呼吸困难属于高风险输入，回答应优先建议及时就医或急救。

In [ ]:
question = "我胸口痛，还有呼吸困难，但不想去医院。"
print(detect_red_flags(question))

answer = review_medical_question(question, store, top_k=1, min_score=0.1)
print(answer.to_json())

validate_medical_citations(answer, [citation_from_chunk(chunks[0])])

## 3. 用药剂量请求拒答

用户要求具体用药或剂量时，系统应拒绝给出处方级建议，并引导咨询医生或药师。

In [ ]:
dosage_answer = review_medical_question(
    "我胸痛但不想去医院，可以吃多少mg止痛药？",
    store,
    top_k=1,
)
dosage_answer.to_dict()

## 4. 生成医学 SFT 样本

医学 SFT 样本必须包含 `not_medical_advice` 边界，并保留 risk tags。

In [ ]:
sft_example = build_medical_sft_example(
    example_id="medical_sft_001",
    source_id="guide_001",
    question=question,
    assistant_output=answer,
    risk_tags=["medical", "not_diagnosis", "needs_human_review"],
)

sft_example.to_dict()

## 5. Model Card 边界

医学项目的 model card 必须明确不替代医生诊断，不提供处方或剂量。

In [ ]:
model_card = ModelCard(
    model_name="medical-qa-assistant",
    version="v1",
    base_model="tiny-base",
    intended_use=["医学科普解释", "危险信号提醒"],
    out_of_scope_use=["不替代医生诊断", "不提供处方或剂量"],
    training_data="medical_sft_v1",
    evaluation="eval_report.md",
    limitations=["不能替代医生", "不能用于急救服务"],
    safety=["危险信号建议及时就医或急救"],
    deployment="local teaching demo",
    owner="course-maintainer",
)

validate_medical_model_card(model_card)
model_card.to_dict()